<a href="https://colab.research.google.com/github/LAB-FAM/nice-rag-project/blob/main/colab/data_ingestion_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALL REQUIRED LIBRARIES
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr
!pip install langchain langchain-community langchain-text-splitters chromadb unstructured[pdf]

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [1]:
import os
import shutil
import subprocess
import time
from langchain_community.document_loaders import DirectoryLoader, UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

In [2]:
# --- FOLDER AND MODEL SETTINGS ---
PDF_DOCS_DIR = "pdf_docs"
CHROMA_DB_DIR = "vector_db/methodology_db"
EMBEDDING_MODEL_NAME = "nomic-embed-text"

In [4]:
# INSTALL AND RUN OLLAMA IN BACKGROUND
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("[*] Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# We will use llama3 as our local reasoning agent
print("[*] Pulling llama3 model (This will take a few minutes)...")
!ollama pull llama3
print("[*] Pulling nomic-embed-text model...")
!ollama pull nomic-embed-text
print("[*] Setup complete!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[*] Starting Ollama server...
[*] Pulling llama3 model (This will take a few minutes)...

[*] Pulling nomic-embed-text model...

[*] Setup complete!


In [5]:
def prepare_vector_database(pdf_dir=PDF_DOCS_DIR, db_dir=CHROMA_DB_DIR, model_name=EMBEDDING_MODEL_NAME):
    """
    Reads PDFs using 'Unstructured' via DirectoryLoader.
    Uses the 'fast' strategy since the goal is to search for DEFINITIONS and RULES rather than tables/SNOMED codes.
    """
    print(f"--- Phase 0: Data Ingestion Started ---")

    if os.path.exists(db_dir):
        print(f"[*] Deleting old database: {db_dir}")
        shutil.rmtree(db_dir)

    if not os.path.exists(pdf_dir) or len(os.listdir(pdf_dir)) == 0:
        print(f"[!] ERROR: Folder '{pdf_dir}' not found or is empty.")
        return

    # Using LangChain DirectoryLoader + UnstructuredPDFLoader
    # This is a more robust way to load a directory of PDFs.
    # It avoids the 'UnstructuredDirectoryLoader' import error.
    print(f"[*] Loading PDF documents by analyzing them with 'Unstructured' (fast mode)...")

    loader = DirectoryLoader(
        pdf_dir,
        glob="**/*.pdf",
        loader_cls=UnstructuredPDFLoader,
        loader_kwargs={"strategy": "fast", "mode": "single"} # 'fast' mode preserves text hierarchy optimally
    )
    documents = loader.load()

    if not documents:
        print("[!] No PDF documents could be read. Cancelling operation.")
        return

    print(f"[*] PDFs successfully processed and structurally loaded.")

    # Text Splitting (Chunking)
    # Since rules and definitions usually span 1-2 paragraphs,
    # chunk_size is optimized to 1500 (approx. 250-300 words).
    print("[*] Splitting structured documents into chunks for the vector database...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=300,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"[*] PDFs split into a total of {len(chunks)} contextual chunks.")

    # 6. Embedding Model and Database Saving
    print(f"[*] Initializing Ollama Embedding Model: '{model_name}'")
    try:
        embeddings = OllamaEmbeddings(model=model_name)
    except Exception as e:
        print(f"[!] Error connecting to Ollama. Error: {e}")
        return

    print(f"[*] Generating embeddings and saving to ChromaDB at: {db_dir}")

    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_dir
    )

    print(f"--- Data Ingestion Complete! ---")
    print(f"[*] Your vector database was successfully saved to the '{db_dir}' folder.")

In [6]:
# Calling the function
prepare_vector_database()

--- Phase 0: Data Ingestion Started ---
[*] Deleting old database: vector_db/methodology_db
[*] Loading PDF documents by analyzing them with 'Unstructured' (fast mode)...


[*] PDFs successfully processed and structurally loaded.
[*] Splitting structured documents into chunks for the vector database...
[*] PDFs split into a total of 565 contextual chunks.
[*] Initializing Ollama Embedding Model: 'nomic-embed-text'
[*] Generating embeddings and saving to ChromaDB at: vector_db/methodology_db


/tmp/ipykernel_25680/2164001117.py:50: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model=model_name)


--- Data Ingestion Complete! ---
[*] Your vector database was successfully saved to the 'vector_db/methodology_db' folder.


In [7]:
# Zip the Database for GitHub Download
print("[*] Zipping the vector database for download...")
if os.path.exists("vector_db"):
    shutil.make_archive("vector_db", 'zip', "vector_db")
    print("[*] Success!")
else:
    print("[!] Error: vector_db folder not found.")

[*] Zipping the vector database for download...
[*] Success!


In [21]:
# Testing the new database
def inspect_database():
    """
    Loads the existing ChromaDB from the disk, checks the total number of chunks,
    and performs a sample similarity search based on a clinical query.
    """
    print("[*] Initializing database inspection...")

    # 1. Check if the database folder actually exists
    if not os.path.exists(CHROMA_DB_DIR):
        print(f"[!] ERROR: Database folder '{CHROMA_DB_DIR}' not found.")
        print("[!] Please ensure you have run the data ingestion script first.")
        return

    # 2. Initialize the Embedding Model
    # We MUST use the exact same model that was used to create the database
    print(f"[*] Loading the embedding model: '{EMBEDDING_MODEL_NAME}'")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

    # 3. Load the existing ChromaDB from disk
    print(f"[*] Loading ChromaDB from: {CHROMA_DB_DIR}")
    vector_db = Chroma(
        persist_directory=CHROMA_DB_DIR,
        embedding_function=embeddings
    )

    # 4. Get and print the total number of chunks stored in the database
    total_chunks = vector_db._collection.count()
    print("\n================ DATABASE INFO ================")
    print(f"Total Contextual Chunks Stored: {total_chunks}")
    print("===============================================\n")

    # 5. Perform a Sample Query
    # Instead of a full patient narrative, we use targeted keywords
    # to specifically extract definitions and management rules from the guidelines.
    sample_query = "Definitions, diagnostic criteria, and clinical management rules for Type 2 Diabetes and Obesity"

    print(f"[*] Performing a similarity search for the targeted query:")
    print(f"    '{sample_query}'\n")

    # k=5 means we want to retrieve the top 5 most relevant chunks to cover both conditions well
    results = vector_db.similarity_search_with_score(sample_query, k=5)

    if not results:
        print("[!] No relevant results found.")
        return

    # 6. Display the Results
    print("================ SEARCH RESULTS ================")
    for index, (document, score) in enumerate(results, start=1):
        # The lower the score (L2 distance), the more relevant the result is
        print(f"--- Result {index} (Similarity Distance: {score:.4f}) ---")

        # Print the source PDF name to know where this info came from
        source_file = document.metadata.get('source', 'Unknown Source')
        # Just getting the filename instead of the full path for cleaner output
        file_name = os.path.basename(source_file)
        print(f"Source PDF: {file_name}")

        # Print the actual text content (we will truncate it a bit for readability)
        content = document.page_content.replace('\n', ' ')

        # Displaying the first 600 characters of the chunk
        print(f"Content Snippet:\n{content[:600]}...\n")

In [20]:
# Run the inspection function
inspect_database()

[*] Initializing database inspection...
[*] Loading the embedding model: 'nomic-embed-text'
[*] Loading ChromaDB from: vector_db/methodology_db

================ DATABASE INFO ================
Total Contextual Chunks Stored: 565

[*] Performing a similarity search for the targeted query:
    'Definitions, diagnostic criteria, and clinical management rules for Type 2 Diabetes and Obesity'

================ SEARCH RESULTS ================
--- Result 1 (Similarity Distance: 299.8434) ---
Source PDF: NG28_type_2_diabetes.pdf
Content Snippet:
about their care, as described in NICE's information on making decisions about your  care.  Making decisions using NICE guidelines explains how we use words to show the  strength (or certainty) of our recommendations, and has information about  prescribing medicines (including off-label use), professional guidelines, standards  and laws (including on consent and mental capacity), and safeguarding.  Healthcare professionals should follow our general gui